<!-- Projeto Desenvolvido na Data Science Academy - www.datascienceacademy.com.br -->
# <font color='blue'>Data Science Academy</font>
## <font color='blue'>Infraestrutura de Dados, IA e Agentes de IA com Databricks</font>
## <font color='blue'>Projeto 6</font>
### <font color='blue'>Deploy de Agentes de IA em Nuvem com Databricks</font>

## Instalação de Pacotes e Dependências

In [0]:
# Instala os pacotes databricks-agents e mlflow silenciosamente
%pip install -U -qqqq databricks-agents>=0.16.0 mlflow>=2.20.2

In [0]:
# Instala o pacote databricks-langchain silenciosamente
%pip install -U -qqqq databricks-langchain

In [0]:
# Instala silenciosamente os pacotes
%pip install -U -qqqq uv langgraph==0.3.4

## Reiniciando a Sessão e Carregando os Pacotes

In [0]:
# Reinicia o ambiente Python no Databricks para aplicar alterações nos pacotes instalados
dbutils.library.restartPython()

In [0]:
# Importa o módulo mlflow para gerenciamento de modelos e experimentos
import mlflow

# Importa tipos necessários para definir interfaces e estados do agente
from typing import Any, Generator, Optional, Sequence, Union

# Importa classes para integração com Databricks usando LangChain
from databricks_langchain import (ChatDatabricks, UCFunctionToolkit, VectorSearchRetrieverTool)

# Importa tipos centrais do LangChain relacionados ao modelo de linguagem e execução de fluxos
from langchain_core.language_models import LanguageModelLike
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool

# Importa componentes do LangGraph para criar fluxos de execução de agentes
from langgraph.graph import END, StateGraph
from langgraph.graph.graph import CompiledGraph
from langgraph.graph.state import CompiledStateGraph

# Importa nó pré-construído de ferramentas para uso em fluxos LangGraph
from langgraph.prebuilt.tool_node import ToolNode

# Importa estado e nó de ferramenta específicos do MLflow para agentes conversacionais
from mlflow.langchain.chat_agent_langgraph import ChatAgentState, ChatAgentToolNode

# Importa classes relacionadas a agentes conversacionais do MLflow
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import (ChatAgentChunk, ChatAgentMessage, ChatAgentResponse, ChatContext)

# Importa o módulo warnings e configura-o para ignorar avisos durante a execução
import warnings
warnings.filterwarnings('ignore')

## Definindo o Endpoint do Modelo de Linguagem no Databricks

In [0]:
# Define o nome do endpoint do modelo de linguagem utilizado no Databricks
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"

## Inicializando o Modelo de Linguagem com o Endpoint 

In [0]:
# Inicializa o modelo de linguagem com o endpoint específico
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

In [0]:
type(llm)

## Definindo Ferramentas Disponíveis no Unity Catalog

In [0]:
# Inicializa uma lista vazia para armazenar ferramentas que o agente pode usar
tools = []

In [0]:
# Define nomes das ferramentas disponíveis no Unity Catalog para serem utilizadas pelo agente
uc_tool_names = ["system.ai.python_exec"]

In [0]:
# Inicializa o toolkit com as ferramentas especificadas no Unity Catalog
uc_toolkit = UCFunctionToolkit(function_names=uc_tool_names)

In [0]:
# Adiciona as ferramentas obtidas do Unity Catalog à lista geral de ferramentas
tools.extend(uc_toolkit.tools)

In [0]:
# Define um prompt de sistema para o comportamento padrão do agente
system_prompt = "Responda com precisão. Se não souber a resposta diga que não sabe ao invés de inventar respostas."

## Construção do Módulo de Execução do Agente

In [0]:
# Define uma função que cria um agente capaz de chamar ferramentas usando LangGraph
def create_tool_calling_agent(
    model: LanguageModelLike,                    # Modelo de linguagem a ser utilizado pelo agente
    tools: Union[ToolNode, Sequence[BaseTool]],  # Ferramentas que o agente pode chamar
    system_prompt: Optional[str] = None,         # Prompt opcional para configurar o comportamento inicial do agente
) -> CompiledGraph:
    # Vincula o modelo às ferramentas, permitindo chamadas automáticas
    model = model.bind_tools(tools)

    # Define uma função que decide qual nó será executado a seguir no fluxo
    def should_continue(state: ChatAgentState):
        
        # Obtém as mensagens atuais do estado do agente
        messages = state["messages"]
        
        # Seleciona a última mensagem recebida
        last_message = messages[-1]
        
        # Se a última mensagem contiver chamadas de ferramentas, continua para o nó de ferramentas
        if last_message.get("tool_calls"):
            return "continue"
        
        # Caso contrário, encerra a execução
        else:
            return "end"

    # Se houver um prompt do sistema definido, adiciona-o antes das mensagens do usuário
    if system_prompt:
        preprocessor = RunnableLambda(
            lambda state: [{"role": "system", "content": system_prompt}]
            + state["messages"]
        )
    # Caso não exista prompt do sistema, mantém as mensagens originais
    else:
        preprocessor = RunnableLambda(lambda state: state["messages"])

    # Combina o pré-processador com o modelo para formar um pipeline de execução
    model_runnable = preprocessor | model

    # Define uma função que chama o modelo com as mensagens processadas
    def call_model(
        state: ChatAgentState,   # Estado atual do agente, incluindo mensagens
        config: RunnableConfig,  # Configuração adicional para execução (opcional)
    ):
        # Invoca o modelo com as mensagens processadas
        response = model_runnable.invoke(state, config)

        # Retorna a resposta do modelo no formato esperado pelo fluxo
        return {"messages": [response]}

    # Cria um fluxo de trabalho utilizando StateGraph para gerenciar estados e transições do agente
    workflow = StateGraph(ChatAgentState)

    # Adiciona o nó que representa o agente (modelo) ao fluxo de trabalho
    workflow.add_node("agent", RunnableLambda(call_model))
    
    # Adiciona o nó que representa as ferramentas que o agente pode utilizar
    workflow.add_node("tools", ChatAgentToolNode(tools))

    # Define o nó "agent" como ponto inicial de entrada do fluxo
    workflow.set_entry_point("agent")

    # Adiciona transições condicionais baseadas no resultado da função should_continue
    workflow.add_conditional_edges(
        "agent",
        should_continue,
        {
            "continue": "tools",  # Se houver chamadas de ferramentas, segue para o nó "tools"
            "end": END,           # Se não houver, termina a execução
        },
    )

    # Define a transição direta do nó "tools" de volta para o "agent" após execução das ferramentas
    workflow.add_edge("tools", "agent")

    # Compila o fluxo criado, tornando-o pronto para uso
    return workflow.compile()

## Construção do Agente com LangGraph

In [0]:
# Define uma classe personalizada que estende ChatAgent para integrar um agente LangGraph
class LangGraphChatAgent(ChatAgent):
    
    # Inicializa a classe com um agente compilado (CompiledStateGraph) do LangGraph
    def __init__(self, agent: CompiledStateGraph):
        
        # Armazena o agente compilado em uma variável de instância
        self.agent = agent

    # Método que gera uma resposta completa com base em mensagens recebidas
    def predict(
        self,
        messages: list[ChatAgentMessage],                # Lista de mensagens recebidas pelo agente
        context: Optional[ChatContext] = None,           # Contexto opcional adicional da conversa
        custom_inputs: Optional[dict[str, Any]] = None,  # Entradas personalizadas opcionais
    ) -> ChatAgentResponse:
        
        # Converte mensagens recebidas para formato de dicionário aceito pelo LangGraph
        request = {"messages": self._convert_messages_to_dict(messages)}

        # Inicializa uma lista para armazenar as mensagens geradas durante a execução
        messages = []
        
        # Itera sobre eventos gerados pela execução do agente (modo streaming: "updates")
        for event in self.agent.stream(request, stream_mode="updates"):
            
            # Itera sobre os dados retornados pelos nós executados no fluxo
            for node_data in event.values():
                
                # Adiciona cada mensagem retornada ao resultado final convertido para ChatAgentMessage
                messages.extend(ChatAgentMessage(**msg) for msg in node_data.get("messages", []))

        # Retorna as mensagens coletadas encapsuladas em um ChatAgentResponse
        return ChatAgentResponse(messages=messages)

    # Método que gera resposta em formato streaming (chunk por chunk)
    def predict_stream(
        self,
        messages: list[ChatAgentMessage],                # Lista de mensagens recebidas pelo agente
        context: Optional[ChatContext] = None,           # Contexto opcional adicional da conversa
        custom_inputs: Optional[dict[str, Any]] = None,  # Entradas personalizadas opcionais
    ) -> Generator[ChatAgentChunk, None, None]:
        
        # Converte mensagens recebidas para formato de dicionário aceito pelo LangGraph
        request = {"messages": self._convert_messages_to_dict(messages)}
        
        # Itera sobre eventos gerados pelo agente em modo de streaming ("updates")
        for event in self.agent.stream(request, stream_mode="updates"):
            
            # Itera sobre os dados retornados pelos nós do fluxo executados
            for node_data in event.values():
                
                # Retorna cada mensagem como um chunk de streaming encapsulado em ChatAgentChunk
                yield from (
                    ChatAgentChunk(**{"delta": msg}) for msg in node_data["messages"]
                )

## Ativando o Registro Automático Para Observabilidade

In [0]:
# Ativa o registro automático das execuções e parâmetros no MLflow usando LangChain
mlflow.langchain.autolog()

## Criando a Instância do Agente

In [0]:
# Cria o agente de IA utilizando o modelo LLM, as ferramentas disponíveis e um prompt do sistema
dsa_agente_ia = create_tool_calling_agent(llm, tools, system_prompt)

In [0]:
type(dsa_agente_ia)

In [0]:
# Inicializa o agente personalizado LangGraphChatAgent com o fluxo criado acima
DSA_AGENTE = LangGraphChatAgent(dsa_agente_ia)

In [0]:
type(DSA_AGENTE)

## Deploy e Observabilidade do Agente de IA

In [0]:
# Enviando uma mensagem simples ao agente para testar o objeto
DSA_AGENTE.predict({"messages": [{"role": "user", "content": "Oi. Testando 123!"}]})

In [0]:
# Executa uma previsão em modo streaming, enviando uma pergunta que será respondida em tempo real pelo agente
for evento in DSA_AGENTE.predict_stream(
    {"messages": [{"role": "user", "content": "Defina o que é investimento no Tesouro Direto no Brasil"}]}
):
    # Imprime cada parte da resposta recebida do agente no modo streaming
    print(evento, "-----------\n")

Este projeto terá continuação no próximo capítulo.

# Fim